In [60]:
import os
import json
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from codex_template import run_codex

In [61]:
load_dotenv(override=True)
api_key=os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)
models = [
    'gpt-5.1-codex',    
    'gpt-5.1-codex-max',     
    'gpt-5.1-chat-latest',  
    'gpt-5.1-2025-11-13'  
]
coding_languages = ['Python', 'C++', 'Rust', 'HTML', 'JavaScript', 'CSS', 'Go', 'Java', 'TypeScript']
languages=['English', 'Spanish', 'German', 'French']

In [62]:
def code_generator_with_comments_tool(model, language, coding_language, message):
    """Generates production-grade code qith using the unified run_codex function"""
    print(f'Code Generator tool called with model: {model}, language: {language} and coding language: {coding_language}')

    full_prompt = f"""
You are a senior software engineer.

TASK:
Generate production-grade, secure {coding_language} code.

USER REQUEST:
{message}

REQUIREMENTS:
- Complete, fully functional code.
- Follow official {coding_language} style guides (e.g., PEP 8 if Python).
- Apply security best practices.
- Include clear docstrings explaining purpose, parameters, and return values in {language} language.
- Add meaningful inline comments explaining design decisions and non-obvious logic.
- Include a short usage example.
- Do NOT include unnecessary verbosity outside the code block.
"""
    return run_codex(full_prompt, model=model, api_key=api_key)

In [63]:
def code_explainer_tool(model, language, coding_language, message):
    """Explains code using the unified run_codex function"""
    print(f'Code Explainer tool called with model: {model}, language: {language} and coding language. {coding_language}')
    
    full_prompt = f"""
    [System: You are an expert code explainer for beginners. Explain this {coding_language} code simply in {language} language.]
    [User: {message}]
    
    Structure:
    1. Simple Summary
    2. Step-by-Step Breakdown (Analogies)
    3. Technical Details
    """
    
    # Use the unified caller!
    return run_codex(full_prompt, model=model, api_key=api_key)

In [64]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "code_generator_with_comments_tool",
            "description": "Generates code based on requirements with comments",
            "parameters": {
                "type": "object",
                "properties": {
                    "model": {"type": "string", "description": "Model to use for generation"},
                    "language": {"type": "string", "description": "Comments language"},
                    "coding_language": {"type": "string", "description": "Programming language"},
                    "message": {"type": "string", "description": "Code requirements"}
                },
                "required": ["model", "language", "coding_language", "message"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "code_explainer_tool",
            "description": "Explains code snippets",
            "parameters": {
                "type": "object",
                "properties": {
                    "model": {"type": "string", "description": "Model to use for explanation"},
                    "language": {"type": "string", "description": "Comments language"},
                    "coding_language": {"type": "string", "description": "Programming language"},
                    "message": {"type": "string", "description": "Code to explain"}
                },
                "required": ["model", "language", "coding_language", "message"]
            }
        }
    }
]

In [65]:
def handle_tool_calls(tool_calls, selected_model, language, coding_language):
    available_tools = {
        "code_generator_with_comments_tool": code_generator_with_comments_tool,
        "code_explainer_tool": code_explainer_tool
    }
    
    tool_responses = []
    
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)
        
        function_args['model'] = selected_model
        function_args['language'] = language
        function_args['coding_language'] = coding_language

        
        if function_name in available_tools:
            function_to_call = available_tools[function_name]
            print(f"Executing {function_name} using {selected_model}...")
            
            result = function_to_call(**function_args)
            
            tool_responses.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": str(result)
            })
    return tool_responses

In [66]:
def code_chatbot(target_model, language, coding_language, message):
    """
    Main function. 
    1. Uses a standard 'Smart' model (gpt-5.1-chat-latest) to understand intent and route tools.
    2. The TOOLS themselves then use the 'target_model' (which might be the specialized Codex).
    """

    router_model = "gpt-5.1-chat-latest" 
    
    print(f" Router ({router_model}) processing request...")
    
    messages = [
        {"role": "system", "content": f"You are a coding assistant. Route the user's request to generate or explain {coding_language} code in {language} language."},
        {"role": "user", "content": message}
    ]
    
    try:
        response = client.chat.completions.create(
            model=router_model,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            max_completion_tokens=500
        )
        
        if response.choices[0].message.tool_calls:
            tool_calls = response.choices[0].message.tool_calls
      
            tool_responses = handle_tool_calls(tool_calls, target_model, language, coding_language)
            return tool_responses[0]["content"]
            
        else:
            return response.choices[0].message.content
    
    except Exception as e:
        return f"Error: {str(e)}\n\nMake sure your API key is valid."

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Codex-Powered Assistant")
    gr.Markdown("Multi-Model AI Coding Agent with routing + Codex execution.")

    with gr.Row():
        model_dropdown = gr.Dropdown(
            label="Target Model",
            choices=models,
            value=models[0],
            scale=1
        )

        language_dropdown = gr.Dropdown(
            label="Response Language",
            choices=languages,
            value="English",
            scale=1
        )

        coding_language_dropdown = gr.Dropdown(
            label="Coding Language",
            choices=coding_languages,
            value="Python",
            scale=1
        )

    with gr.Row():
        with gr.Column(scale=1):
            message_input = gr.Textbox(
                label="Request",
                placeholder="Example: Create a snake game or Explain this code...",
                lines=20,   # caja grande
                show_label=True
            )
            submit_btn = gr.Button("Generate", variant="primary")

        with gr.Column(scale=1):
            response_output = gr.Markdown(
                label="Response",
                elem_id="response_box"
            )

    submit_btn.click(
        fn=code_chatbot,
        inputs=[model_dropdown, language_dropdown, coding_language_dropdown, message_input],
        outputs=response_output
    )

if __name__ == "__main__":
    demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7871
* Running on public URL: https://5c34f3a0c5ed71f743.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


 Router (gpt-5.1-chat-latest) processing request...
Executing code_generator_with_comments_tool using gpt-5.1-codex...
Code Generator tool called with model: gpt-5.1-codex, language: English and coding language: Python
Generating code using gpt-5.1-codex...
